# Helsinki Walkability & Population Density Prediction
**Data:** OpenStreetMap (road network + POIs) · HSY Population Grid (250m cells)  
**Models:** Random Forest Regression (population density) + Classification (Dense/Medium/Sparse)

In [ ]:
# Install GIS libraries (run once, then restart kernel)
# %pip install osmnx geopandas shapely

In [ ]:

import osmnx as ox                                          # download OSM map data
import geopandas as gpd                                     # work with geographic data
import pandas as pd                                         # data manipulation
import numpy as np                                          # numerical operations
import matplotlib.pyplot as plt                             # plotting
import warnings                                             # suppress non-critical warnings
from shapely.geometry import Point                          # create point geometries
from sklearn.model_selection import train_test_split        # split data into train/test
from sklearn.preprocessing import StandardScaler            # normalize feature values
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier  # ML models
from sklearn.metrics import mean_absolute_error, r2_score, classification_report, confusion_matrix  # evaluation
from matplotlib.patches import Patch                        # for custom map legend

warnings.filterwarnings('ignore')                           # hide warnings for cleaner output
print(f"osmnx {ox.__version__} | geopandas {gpd.__version__}")  # confirm versions

In [ ]:
# Download Helsinki road network and visualize

G = ox.graph_from_place("Helsinki, Finland", network_type="walk")  # download walkable roads
helsinki_boundary = ox.geocode_to_gdf("Helsinki, Finland")          # get city boundary polygon
print(f"Nodes (intersections): {len(G.nodes)} | Edges (streets): {len(G.edges)}")

fig, ax = plt.subplots(figsize=(10, 8))                             # create figure
ox.plot_graph(G, ax=ax, node_size=0, edge_linewidth=0.3,            # plot road network
              edge_color='steelblue', bgcolor='white', show=False)
helsinki_boundary.boundary.plot(ax=ax, color='red', linewidth=2)    # overlay city boundary
ax.set_title("Helsinki Walkable Road Network (OSM)", fontsize=14)   # chart title
plt.tight_layout()
plt.show()

In [ ]:
# Download Points of Interest (POIs) from OSM

place = "Helsinki, Finland"                                          # area to query
poi_tags = {                                                         # OSM tags to fetch
    "transit":  {"public_transport": "stop_position"},              # bus/tram stops
    "shop":     {"shop": True},                                      # all shops
    "food":     {"amenity": ["restaurant", "cafe", "fast_food"]},   # food places
    "school":   {"amenity": "school"},                               # schools
    "park":     {"leisure": "park"},                                 # parks
}

poi_list = []                                                        # collect all POIs here
for ptype, tags in poi_tags.items():                                 # loop over each category
    try:
        gdf = ox.features_from_place(place, tags=tags)              # fetch from OSM
        gdf["poi_type"] = ptype                                      # label category
        poi_list.append(gdf[["geometry", "poi_type"]])              # keep only needed columns
        print(f"  {ptype}: {len(gdf)} features")
    except Exception as e:
        print(f"  {ptype}: skipped ({e})")

pois = pd.concat(poi_list, ignore_index=True)                       # merge all into one GDF
pois = gpd.GeoDataFrame(pois, geometry="geometry", crs="EPSG:4326") # set coordinate system
pois["geometry"] = pois["geometry"].centroid                        # convert polygons to points
print(f"\nTotal POIs: {len(pois)}")

In [ ]:
# Load Helsinki population grid
# Download 2023 SHP from: https://avoidatastr.blob.core.windows.net/avoindata/AvoinData/6_Asuminen/Vaestotietoruudukko/Shp/Vaestotietoruudukko_2023_shp.zip
# Extract the zip and place all files in the same folder as this notebook

FILE = "Vaestotietoruudukko_2023.shp"                                # shapefile name after extracting zip
pop_raw = gpd.read_file(FILE)                                        # load shapefile into GeoDataFrame
pop_raw = pop_raw.to_crs(epsg=4326)                                  # reproject from ETRS-GK25 to WGS84
print("Columns:", list(pop_raw.columns))                             # check column names
pop_col = [c for c in pop_raw.columns if "ASUKKAITA" in c.upper()][0]  # find the residents column
pop_grid = pop_raw[["geometry", pop_col]].rename(columns={pop_col: "population"})  # keep and rename
pop_grid["population"] = pd.to_numeric(pop_grid["population"], errors="coerce")     # ensure numeric
pop_grid = pop_grid[pop_grid["population"] > 0].reset_index(drop=True)  # remove empty cells
pop_grid["centroid"] = pop_grid.geometry.centroid                    # precompute cell centroids
print(f"Loaded {len(pop_grid)} cells | pop range: {pop_grid.population.min():.0f}–{pop_grid.population.max():.0f}")
pop_grid.head()                                                      # preview

In [ ]:
# Feature engineering: count POIs and street nodes near each grid cell
# Uses spatial joins (fast) instead of looping over each cell

EPSG_METRIC = 3879                                                   # Finnish projected CRS (meters)
BUFFER_M = 500                                                       # search radius in meters

# Build buffer zones: project centroids to metric CRS, then buffer 500m
grid_buf = pop_grid.copy()                                           # copy to avoid modifying original
grid_buf = grid_buf.set_geometry("centroid")                         # use centroid as active geometry
grid_buf = grid_buf.to_crs(epsg=EPSG_METRIC)                         # project to meters
grid_buf["geometry"] = grid_buf.geometry.buffer(BUFFER_M)            # create 500m buffer circles
grid_buf = grid_buf.set_geometry("geometry")                         # set buffer as active geometry

pois_proj = pois.to_crs(epsg=EPSG_METRIC)                           # project POIs to same CRS

# Count each POI type within the buffer using spatial join
for ptype in pois["poi_type"].unique():                              # loop each category
    subset = pois_proj[pois_proj["poi_type"] == ptype]              # filter to one POI type
    joined = gpd.sjoin(grid_buf[["geometry"]], subset[["geometry"]],  # spatial join: POIs inside buffer
                       how="left", predicate="contains")
    pop_grid[f"{ptype}_count"] = (                                   # count per cell
        joined.groupby(joined.index).size().reindex(pop_grid.index, fill_value=0))

# Count street intersections (graph nodes) within buffer
nodes, _ = ox.graph_to_gdfs(G)                                      # extract nodes as GDF
nodes_proj = nodes[["geometry"]].to_crs(epsg=EPSG_METRIC)           # project nodes to meters
joined_nodes = gpd.sjoin(grid_buf[["geometry"]], nodes_proj,         # spatial join with street nodes
                          how="left", predicate="contains")
pop_grid["street_count"] = (                                         # count intersections per cell
    joined_nodes.groupby(joined_nodes.index).size().reindex(pop_grid.index, fill_value=0))

# Compute total POI count across all categories
poi_cols = [f"{t}_count" for t in pois["poi_type"].unique()]        # list of POI count columns
pop_grid["poi_total"] = pop_grid[poi_cols].sum(axis=1)              # sum across all POI types

FEATURES = poi_cols + ["street_count", "poi_total"]                 # full feature list
print("Features created:", FEATURES)
print(pop_grid[FEATURES + ["population"]].describe().round(1))      # quick stats

In [ ]:
# Exploratory Data Analysis: feature distributions

cols = FEATURES + ["population"]                                     # columns to plot
fig, axes = plt.subplots(3, 4, figsize=(14, 9))                      # 3x4 grid of subplots
axes = axes.flatten()                                                # flatten to 1D list

for i, col in enumerate(cols):                                       # one histogram per feature
    axes[i].hist(pop_grid[col].dropna(), bins=30,                    # plot distribution
                 color='steelblue', edgecolor='white')
    axes[i].set_title(col, fontsize=10)                              # column name as title
    axes[i].set_xlabel("Value")                                      # x-axis label

for j in range(len(cols), len(axes)):                                # hide unused subplots
    axes[j].set_visible(False)

plt.suptitle("Feature Distributions — Helsinki Grid Cells", fontsize=13)
plt.tight_layout()
plt.show()

# Correlation heatmap
corr = pop_grid[cols].corr()                                         # compute correlation matrix
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)              # show as colored grid
plt.colorbar(im, ax=ax)                                             # color scale bar
ax.set_xticks(range(len(cols)))                                      # set x tick positions
ax.set_yticks(range(len(cols)))                                      # set y tick positions
ax.set_xticklabels(cols, rotation=45, ha="right", fontsize=8)       # x labels rotated
ax.set_yticklabels(cols, fontsize=8)                                 # y labels
ax.set_title("Feature Correlation Matrix")                           # title
plt.tight_layout()
plt.show()

In [ ]:
# Train Regression and Classification Models

# Prepare data 
df = pop_grid[FEATURES + ["population"]].dropna()                   # drop rows missing any value
X = df[FEATURES].values                                             # feature matrix (numeric array)
y_reg = df["population"].values                                     # regression target: population count

p33 = np.percentile(y_reg, 33)                                      # lower threshold (33rd percentile)
p66 = np.percentile(y_reg, 66)                                      # upper threshold (66th percentile)
y_clf = np.where(y_reg <= p33, "Sparse",                            # below p33 = Sparse
         np.where(y_reg <= p66, "Medium", "Dense"))                 # below p66 = Medium, else Dense
print(f"Classes: Sparse ≤{p33:.0f} | Medium ≤{p66:.0f} | Dense >{p66:.0f} residents")
print(pd.Series(y_clf).value_counts().to_string())                  # show class balance

# Train/test split
X_tr, X_te, yr_tr, yr_te, yc_tr, yc_te = train_test_split(         # split once, reuse for both tasks
    X, y_reg, y_clf, test_size=0.2, random_state=42)

scaler = StandardScaler()                                           # scale features to mean=0, std=1
X_tr = scaler.fit_transform(X_tr)                                   # fit on train, transform train
X_te = scaler.transform(X_te)                                       # transform test (same scale)

# Regression model 
reg = RandomForestRegressor(n_estimators=100, max_depth=10,          # 100 trees, max depth 10
                             random_state=42, n_jobs=-1)             # use all CPU cores
reg.fit(X_tr, yr_tr)                                                # train on training data
yr_pred = reg.predict(X_te)                                         # predict on test data
print(f"\nRegression — MAE: {mean_absolute_error(yr_te, yr_pred):.1f} | R²: {r2_score(yr_te, yr_pred):.3f}")

# Classification model 
clf = RandomForestClassifier(n_estimators=100, max_depth=10,         # same structure as regressor
                              random_state=42, n_jobs=-1)
clf.fit(X_tr, yc_tr)                                                # train classifier
yc_pred = clf.predict(X_te)                                         # predict classes
print("\nClassification Report:")
print(classification_report(yc_te, yc_pred))                        # precision, recall, f1 per class

# Feature importance plot (regression) 
imp = pd.Series(reg.feature_importances_, index=FEATURES).sort_values()  # importance per feature
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
imp.plot(kind="barh", ax=axes[0], color="steelblue")               # horizontal bar chart
axes[0].set_title("Feature Importance (Regression)")
axes[0].set_xlabel("Importance Score")

# Actual vs Predicted plot (regression) 
axes[1].scatter(yr_te, yr_pred, alpha=0.3, s=15, color="steelblue") # scatter of predictions
mx = max(yr_te.max(), yr_pred.max())                                # max value for reference line
axes[1].plot([0, mx], [0, mx], 'r--', label="Perfect prediction")  # diagonal = perfect
axes[1].set_xlabel("Actual Population")
axes[1].set_ylabel("Predicted Population")
axes[1].set_title("Regression: Actual vs Predicted")
axes[1].legend()
plt.tight_layout()
plt.show()

# Confusion matrix (classification) 
classes = ["Dense", "Medium", "Sparse"]                             # class order
cm = confusion_matrix(yc_te, yc_pred, labels=classes)              # compute matrix
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")                                    # blue heatmap
plt.colorbar(im, ax=ax)                                             # color scale
ax.set_xticks(range(3)); ax.set_yticks(range(3))                    # tick positions
ax.set_xticklabels(classes); ax.set_yticklabels(classes)           # class labels
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")                # axis labels
ax.set_title("Confusion Matrix — Classification")                   # title
for i in range(3):                                                  # add count numbers inside each cell
    for j in range(3):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize predictions across Helsinki

valid = pop_grid[FEATURES].dropna()                                  # only cells with all features
X_all = scaler.transform(valid.values)                              # scale using the same scaler
plot_gdf = pop_grid.loc[valid.index].copy()                         # matching rows from grid
plot_gdf["pred_pop"] = reg.predict(X_all)                          # regression predictions
plot_gdf["pred_class"] = clf.predict(X_all)                        # classification predictions

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Left: Population density heatmap
plot_gdf.plot(column="pred_pop", cmap="YlOrRd", ax=axes[0],         # yellow-orange-red gradient
              legend=True, legend_kwds={"label": "Predicted Residents", "shrink": 0.6})
axes[0].set_title("Predicted Population Density (Regression)", fontsize=13)
axes[0].set_axis_off()                                              # hide axes ticks

# Right: Neighborhood type map
color_map = {"Dense": "#d73027", "Medium": "#fee08b", "Sparse": "#1a9850"}  # class colors
plot_gdf.plot(color=plot_gdf["pred_class"].map(color_map), ax=axes[1])      # color by class
legend_elements = [Patch(facecolor=v, label=k) for k, v in color_map.items()]  # legend patches
axes[1].legend(handles=legend_elements, loc="lower right", fontsize=11)     # add legend
axes[1].set_title("Predicted Neighborhood Type (Classification)", fontsize=13)
axes[1].set_axis_off()                                              # hide axes ticks

plt.suptitle("Helsinki ML Predictions — OSM Features | Target: HSY Population Grid 2023", fontsize=13)
plt.tight_layout()
plt.show()